# Reporte ejecutivo diario — cumplimiento del precio de lista

Una sola lámina, a **corte de ayer** contra el mes anterior completo, para las dos regiones, que responde:

1. **¿En qué canal de venta se cumple menos el precio de lista?** Y dentro de ese canal, ¿en qué canal de distribución
   y en qué grupo de material 6?
2. **¿Cuál es el cumplimiento general a corte del día?** Por región y total compañía, con la variación frente al mes
   anterior, y el cumplimiento por tipo de descuento (ajuste de precio de mercado y fecha corta).

Además genera **una lámina por región** (Región Oriente y Región Occidente) con más detalle: por qué se
descuenta, dónde (canal de venta × canal de distribución) y la tendencia del cumplimiento acumulado del mes
contra el mes anterior (C.3).

Ejecute todo en orden (*Run All*). Consulta la base con `consultas.descargar_mes()` (la misma consulta del reporte
diario y del semanal) y procesa en memoria, sin Parquet. El cumplimiento es precio neto / precio de lista (A.3).

# A. Preparación
## A.1 Parámetros

> **Siempre a corte de ayer** contra el mes anterior completo (A.4). `PARTICIPACION_MIN`: el canal de distribución
> y el grupo de material 6 señalados son los de menor cumplimiento entre los que pesan al menos ese % de la venta
> bruta de su nivel.

In [19]:
from datetime import date, timedelta
from pathlib import Path

AQUI = Path.cwd()
SALIDA = AQUI / "output" / "Reporte_Ejecutivo"   # carpeta donde se guardan los resultados de este notebook
SALIDA.mkdir(parents=True, exist_ok=True)
CANALES_VENTA_INFORME = ["Institucional", "Amas de casa", "Modernos", "Tradicional"]   # canales de venta que se comparan
PARTICIPACION_MIN = 5    # % minimo de la venta bruta de su nivel para elegir el canal de distribucion y el grupo de
                         # material 6 con menor cumplimiento (evita senalar grupos con muy poca venta)

## A.2 Librerías y conexión

Usa la conexión `DB_*` del `.env` de esta carpeta o, si no hay, el del proyecto de lista de precios.

In [20]:
import base64
import io
import os
from contextlib import contextmanager

import matplotlib.pyplot as plt
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
from dotenv import load_dotenv
from IPython.display import HTML, Markdown, display
from sqlalchemy import URL, create_engine, text

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)


def engine():
    load_dotenv(AQUI / ".env") or load_dotenv(AQUI.parent / "Lista de precios" / ".env")
    faltan = [v for v in ("DB_HOST", "DB_NAME", "DB_USER", "DB_PASSWORD") if not os.getenv(v)]
    if faltan:
        raise RuntimeError(f"Faltan en el .env: {', '.join(faltan)}. "
                           f"Copie .env.example como .env en {AQUI} y completelo.")
    url = URL.create("postgresql+psycopg2", username=os.environ["DB_USER"], password=os.environ["DB_PASSWORD"],
                     host=os.environ["DB_HOST"], port=int(os.getenv("DB_PORT", "5432")),
                     database=os.environ["DB_NAME"])
    return create_engine(url, pool_pre_ping=True)


ENGINE = engine()
with ENGINE.connect() as c:
    c.execute(text("SELECT 1"))
print(f"Conexion OK: {ENGINE.url.host}/{ENGINE.url.database}")

Conexion OK: 192.168.13.93/commercial_bi


## A.3 Consulta (consultas.py)

La consulta y los cálculos por fila están en `consultas.py` (`SQL_REPORTES`, `calcular()`, `descargar_mes()`), que
comparten el reporte diario y el semanal:

- **Devolución:** línea con `kilogram < 0` y `value < 0`. Las demás son venta, incluidas las de valor negativo sin
  kilos (p. ej. notas crédito).
- **Kilos brutos** (`Venta Kg`) = `kilogram` sin devoluciones; **kilos devueltos** (`Devolucion Kg`, en positivo);
  **kilos netos** = `SUM(kilogram)` de todo = kilos brutos − kilos devueltos.
- **Valor bruto** (`Venta Bruta`) = `value + discount_value` sin devoluciones; **valor de devolución**
  (`Devolucion Valor`, en positivo) = `value + discount_value` de las devoluciones; **valor ajuste de precio**
  (`Valor Total Descuento`) = `SUM(discount_value)` de todo; **valor neto** (`Venta Neta`) = `SUM(value)` de todo.
  Valor bruto − devolución − ajuste = valor neto.
- **Precio de lista** = valor bruto / kilos brutos; **precio neto** = valor neto / kilos netos; **% cumplimiento del
  precio de lista** = precio neto / precio de lista. Por tipo de ajuste (mercado, fecha corta) sigue sobre la venta
  bruta, como si fuera el único: (valor bruto − ese ajuste) / valor bruto.
- Fecha corta = `short_dated_discount`; ajuste de precio de mercado = `discount_value − short_dated_discount`.
- Grano: fecha × cliente × grupo de material 6 × distribuidora × canal × sector. El % de lista y el rango (22
  tramos) se calculan sobre ese grano.

In [21]:
# El SQL, los calculos (% de lista, rangos, motivo) y la descarga estan en consultas.py: los comparten el reporte
# diario y el semanal.
from consultas import FILAS_POR_PARTE, _ROTULO_SIN_BRUTA, _ROTULOS, descargar_mes

## A.4 Periodo

Del 1 del mes anterior (completo) hasta `CORTE_D` (incluido): en el reporte diario, **siempre hasta ayer**.

In [22]:
CORTE_D = date.today() - timedelta(days=1)                       # el diario siempre va hasta ayer
INICIO_ACT = CORTE_D.replace(day=1)                              # 1 del mes en curso
INICIO_ANT = (INICIO_ACT - timedelta(days=1)).replace(day=1)     # 1 del mes anterior
FIN_D = CORTE_D + timedelta(days=1)                              # EXCLUSIVO
print(f"Periodo: {INICIO_ANT:%d/%m/%Y} (mes anterior completo) a {CORTE_D:%d/%m/%Y} (incluido)")

Periodo: 01/09/2026 (mes anterior completo) a 26/10/2026 (incluido)


## A.5 Nombres, formato y funciones comunes

In [23]:
SOCIEDADES = {"1000": "Región Oriente", "2000": "Región Occidente"}   # company_code: 1000 Región Oriente, 2000 Región Occidente
VISTAS = {"Región Oriente": "Región Oriente", "Región Occidente": "Región Occidente"}
NOMBRE_CANAL_VENTA = {"Directo": "Amas de casa"}
MEDIDAS = ["Venta Kg", "Venta Bruta", "Venta Neta", "Valor Total Descuento",
           "Descuento Ajuste Precio Mercado", "Descuento Rotacion / Fecha Corta",
           "Devolucion Kg", "Devolucion Valor"]   # brutos sin devoluciones; neto y ajuste total de todo (A.3)

# --- nombres normalizados
VENTA_BRUTA, VENTA_NETA, VENTA_KG = "Venta bruta ($)", "Venta neta ($)", "Venta kg"
PRECIO_LISTA, PRECIO_NETO = "Precio lista por kg", "Precio neto por kg"
L_GEN, D_GEN = "% cumplimiento del precio de lista general", "% Ajuste general"
L_AJ, D_AJ = "% cumplimiento del precio de lista ajuste de precio de mercado", "% Ajuste de precio de mercado"
L_FC, D_FC = "% cumplimiento del precio de lista fecha corta", "% Ajuste de fecha corta"
P_GEN, P_AJ, P_FC = "Ajuste general ($)", "Ajuste de precio de mercado ($)", "Ajuste de fecha corta ($)"
MILLON = 1     # divisor de los montos: 1 = pesos completos (1e6 = millones)
NOTA_CUMPLIMIENTO = ("% cumplimiento = precio neto / precio de lista (precio neto = valor neto / kilos netos, con "
                     "devoluciones; precio de lista = valor bruto / kilos brutos, sin devoluciones)")   # pie de las laminas
COLOR_SOC = {"Región Oriente": "#2a78d6", "Región Occidente": "#eb6834"}
MESES_ES = ["enero", "febrero", "marzo", "abril", "mayo", "junio", "julio", "agosto",
            "septiembre", "octubre", "noviembre", "diciembre"]


# --- formato colombiano
def num(v, dec: int = 1) -> str:
    """Miles con punto, decimales con coma (3.831,3)."""
    if pd.isna(v):
        return "-"
    return f"{v:,.{dec}f}".replace(",", "_").replace(".", ",").replace("_", ".")


def pct(v) -> str:
    return "-" if pd.isna(v) else num(v) + "%"


def variacion(v, en_puntos: bool) -> str:
    """+5,9% (variacion) o -6,4 pts (diferencia entre porcentajes)."""
    if pd.isna(v):
        return "-"
    return ("+" if v >= 0 else "-") + num(abs(v)) + (" pts" if en_puntos else "%")


# --- indicadores (todo desde sumas, nunca promediando % de filas)
def indicadores(s: pd.DataFrame) -> pd.DataFrame:
    """s: sumas de MEDIDAS (cualquier indice) -> BLOQUE de indicadores (ver A.3): % cumplimiento general = precio neto
    / precio de lista; por tipo de ajuste, sobre la venta bruta, como si fuera el unico."""
    bruta = s["Venta Bruta"]
    b = bruta.where(bruta != 0)
    kg = s["Venta Kg"].where(s["Venta Kg"] != 0)
    kg_neto = s["Venta Kg"] - s["Devolucion Kg"]                   # kilos netos: SUM(kilogram) de todo
    precio_lista = bruta / kg
    precio_neto = s["Venta Neta"] / kg_neto.where(kg_neto > 0)     # SUM(value) / SUM(kilogram)
    aj, fc = s["Descuento Ajuste Precio Mercado"], s["Descuento Rotacion / Fecha Corta"]
    gen = s["Valor Total Descuento"]                               # SUM(discount_value) de todo
    return pd.DataFrame({
        VENTA_BRUTA: bruta / MILLON,
        VENTA_NETA: s["Venta Neta"] / MILLON,
        VENTA_KG: s["Venta Kg"],
        PRECIO_LISTA: precio_lista,
        PRECIO_NETO: precio_neto,
        L_GEN: precio_neto / precio_lista.where(precio_lista > 0) * 100,
        D_GEN: gen / b * 100,
        P_GEN: gen / MILLON,
        L_AJ: (bruta - aj) / b * 100,
        D_AJ: aj / b * 100,
        P_AJ: aj / MILLON,
        L_FC: (bruta - fc) / b * 100,
        D_FC: fc / b * 100,
        P_FC: fc / MILLON,
    }, index=s.index)


def ejes_limpios(ax, formato_y=None, formato_x=None):
    ax.grid(axis="y" if formato_y else "x", color="#e6e5e1", linewidth=0.8)
    ax.set_axisbelow(True)
    for lado in ("top", "right"):
        ax.spines[lado].set_visible(False)
    ax.tick_params(colors="#52514e", labelsize=8)
    if formato_y:
        ax.yaxis.set_major_formatter(formato_y)
    if formato_x:
        ax.xaxis.set_major_formatter(formato_x)

## A.6 Consulta y agregación

Consulta el mes anterior y el mes en curso y deja `base`: las sumas por sociedad × canal de venta × canal de
distribución × grupo de material 6 × periodo. Todos los % se calculan desde esas sumas (`indicadores()`).

`diario` (sociedad × día) trae además los kilos netos para las barras de la lámina 2: `Venta Neta Kg` =
`Venta Kg − Devolucion Kg` (las devoluciones vienen en positivo). El valor neto ya es `Venta Neta` (`SUM(value)`).

In [24]:
COLUMNAS_EJECUTIVO = ["Fecha", "Sociedad", "Canal de Venta", "Canal de Distribucion", "Nombre GR6", *MEDIDAS]
NIVELES = ["Sociedad", "Canal de Venta", "Canal de Distribucion", "Nombre GR6", "Periodo"]

print(f"Consultando {INICIO_ANT} -> {FIN_D} (exclusivo)...")
datos = descargar_mes(INICIO_ANT, FIN_D, columnas=COLUMNAS_EJECUTIVO, motor=ENGINE)
FECHA_MAX_D = datos["Fecha"].max()
MES_ANT_TXT, MES_ACT_TXT = MESES_ES[INICIO_ANT.month - 1], MESES_ES[INICIO_ACT.month - 1]
ETQ_ANT = f"{MES_ANT_TXT[:3].capitalize()} {INICIO_ANT.year}"          # "Sep 2026"
ETQ_ACT = f"{MES_ACT_TXT[:3].capitalize()} al {FECHA_MAX_D.day}"       # "Oct al 21"

datos["Sociedad"] = datos["Sociedad"].map(SOCIEDADES).fillna(datos["Sociedad"]).fillna("Sin sociedad")
datos = datos[datos["Sociedad"].isin(VISTAS.values())]
datos["Canal de Venta"] = datos["Canal de Venta"].replace(NOMBRE_CANAL_VENTA)
datos = datos.fillna({"Canal de Venta": "(sin canal)", "Canal de Distribucion": "(vacío)", "Nombre GR6": "(vacío)"})
datos["Periodo"] = pd.Series(ETQ_ANT, index=datos.index).mask(datos["Fecha"] >= pd.Timestamp(INICIO_ACT), ETQ_ACT)
base = datos.groupby(NIVELES)[MEDIDAS].sum()   # sociedad x canal de venta x canal de distribucion x grupo 6 x periodo
diario = datos.groupby(["Sociedad", "Fecha"])[MEDIDAS].sum()   # sociedad x dia (tendencia de las laminas por region)
# kilos netos (las devoluciones vienen en positivo): barras de la lamina 2
diario["Venta Neta Kg"] = diario["Venta Kg"] - diario["Devolucion Kg"]
del datos
print(f"Datos hasta {FECHA_MAX_D:%d/%m/%Y} · {ETQ_ANT} vs {ETQ_ACT} | {len(base):,} combinaciones")

Consultando 2026-09-01 -> 2026-10-27 (exclusivo)...
  500,000 filas
  767,147 filas
Datos hasta 26/10/2026 · Sep 2026 vs Oct al 26 | 189 combinaciones


# C. Salidas

`Informe Diario Cumplimiento precio lista <mes anterior> vs <mes en curso> al <día>.pptx` (total y las dos
regiones) y `Informe Diario Cumplimiento precio lista <región> <mes anterior> vs <mes en curso> al <día>.pptx`
(una por región), en `SALIDA`.

## C.1 Funciones comunes de PowerPoint

In [25]:
from pptx import Presentation
from pptx.dml.color import RGBColor
from pptx.enum.text import MSO_ANCHOR, PP_ALIGN
from pptx.util import Inches, Pt


def rgb(h: str) -> RGBColor:
    return RGBColor.from_string(h.lstrip("#").upper())


TINTA, TINTA_2, TINTA_3 = rgb("#0b0b0b"), rgb("#52514e"), rgb("#8c8b87")
ANCHO, ALTO, MARGEN = Inches(13.333), Inches(7.5), Inches(0.5)
UTIL_W = ANCHO - 2 * MARGEN
CUERPO_Y = Inches(1.6)
CUERPO_H = ALTO - CUERPO_Y - Inches(0.55)
prs = None   # presentacion en construccion


def nueva_presentacion():
    global prs
    prs = Presentation()
    prs.slide_width, prs.slide_height = ANCHO, ALTO


def guardar(nombre: str):
    archivo_ = SALIDA / nombre
    try:
        prs.save(archivo_)
        print(f"{archivo_.name}: {len(prs.slides)} láminas")
    except PermissionError:
        print(f"No se pudo guardar {archivo_.name}: ciérrelo en PowerPoint y vuelva a correr esta celda.")


def nombre_informe(region: str = "") -> str:
    """'Informe Diario Cumplimiento precio lista [región] Sep 2026 vs Oct 2026 al 21.pptx'."""
    partes = ["Informe Diario Cumplimiento precio lista", region, ETQ_ANT,
              f"vs {MES_ACT_TXT[:3].capitalize()} {INICIO_ACT.year} al {FECHA_MAX_D.day}"]
    return " ".join(p for p in partes if p) + ".pptx"


def caja_texto(d, x, y, w, h, parrafos, tam=14, color=TINTA, negrita=False, alinear=PP_ALIGN.LEFT, espacio=0):
    """parrafos: texto o lista; cada parrafo es texto o lista de (texto, negrita)."""
    caja = d.shapes.add_textbox(x, y, w, h)
    tf = caja.text_frame
    tf.word_wrap = True
    for i, par in enumerate([parrafos] if isinstance(parrafos, str) else parrafos):
        p = tf.paragraphs[0] if i == 0 else tf.add_paragraph()
        p.alignment = alinear
        p.space_after = Pt(espacio)
        for trozo, neg in ([(par, negrita)] if isinstance(par, str) else par):
            r = p.add_run()
            r.text = trozo
            r.font.size, r.font.bold = Pt(tam), neg
            r.font.color.rgb = color
    return caja


def imagen(d, fig, x=MARGEN, y=CUERPO_Y, w=UTIL_W, h=CUERPO_H):
    """Figura ajustada a la caja w x h, sin deformarla y centrada en horizontal."""
    buf = io.BytesIO()
    fig.savefig(buf, format="png", dpi=170, bbox_inches="tight", facecolor="white")
    plt.close(fig)
    buf.seek(0)
    pic = d.shapes.add_picture(buf, x, y)
    escala = min(w / pic.width, h / pic.height)
    pic.width, pic.height = int(pic.width * escala), int(pic.height * escala)
    pic.left = int(x + (w - pic.width) / 2)
    return pic

## C.2 Lámina ejecutiva

Arriba, el total compañía. Una mitad por región: el **cumplimiento general** a corte con su variación, el
cumplimiento por **canal de venta** (el menor, resaltado), la **ruta** canal de venta → canal de distribución → grupo de
material 6 donde menos se cumple y el **cumplimiento por tipo de descuento** (ajuste de precio de mercado y fecha
corta, cada uno como si fuera el único descuento).

In [26]:
MEJORA, BAJA = rgb("#1baf7a"), rgb("#d64545")   # el cumplimiento sube / baja frente al mes anterior
FONDO_RUTA = rgb("#f6f5f2")


def resumen(filtro: dict, por=None) -> pd.DataFrame:
    """Cumplimiento del mes en curso y del anterior, venta bruta y ajuste del mes en curso. filtro: {nivel: valor};
    por: nivel que se abre (None = una sola fila con el total del filtro). Todo desde sumas (indicadores())."""
    b = base
    for nivel, valor in filtro.items():
        b = b[b.index.get_level_values(nivel) == valor]
    ind = indicadores(b.groupby(level=[por, "Periodo"] if por else "Periodo").sum())
    if not por:
        ind = pd.concat({"Total": ind}, names=["Miembro"])
    act = ind[ind.index.get_level_values("Periodo") == ETQ_ACT].droplevel("Periodo")
    ant = ind[ind.index.get_level_values("Periodo") == ETQ_ANT].droplevel("Periodo").reindex(act.index)
    t = pd.DataFrame({"cumpl": act[L_GEN], "cumpl_ant": ant[L_GEN], "bruta": act[VENTA_BRUTA], "ajuste": act[P_GEN]})
    t["delta"] = t["cumpl"] - t["cumpl_ant"]
    t["part"] = t["bruta"] / t["bruta"].sum() * 100
    return t


def menor_cumplimiento(t: pd.DataFrame):
    """El miembro con menor cumplimiento entre los que pesan al menos PARTICIPACION_MIN % de la venta bruta."""
    t = t[(t["bruta"] > 0) & (t["part"] >= PARTICIPACION_MIN)]
    return t["cumpl"].idxmin()


def ruta(soc):
    """Canal de venta con menor cumplimiento; dentro, el canal de distribucion; dentro, el grupo de material 6."""
    cv_t = resumen({"Sociedad": soc}, "Canal de Venta").reindex(CANALES_VENTA_INFORME)
    cv = cv_t["cumpl"].idxmin()
    cd_t = resumen({"Sociedad": soc, "Canal de Venta": cv}, "Canal de Distribucion")
    cd = menor_cumplimiento(cd_t)
    g_t = resumen({"Sociedad": soc, "Canal de Venta": cv, "Canal de Distribucion": cd}, "Nombre GR6")
    g = menor_cumplimiento(g_t)
    return cv_t, (cv, cv_t.loc[cv]), (cd, cd_t.loc[cd]), (g, g_t.loc[g])


def cumplimiento_por_tipo(soc) -> list:
    """% cumplimiento del precio de lista de cada tipo de descuento (como si fuera el unico): mes en curso y anterior."""
    ind = indicadores(base[base.index.get_level_values("Sociedad") == soc].groupby(level="Periodo").sum())
    return [("ajuste de precio de mercado", ind.loc[ETQ_ACT, L_AJ], ind.loc[ETQ_ANT, L_AJ], "#1baf7a"),
            ("fecha corta", ind.loc[ETQ_ACT, L_FC], ind.loc[ETQ_ANT, L_FC], "#eda100")]


def millones(v) -> str:
    return "-" if pd.isna(v) else "$" + num(v / 1e6, 0) + " M"


def figura_canales(cv_t: pd.DataFrame, peor, general, color, figsize=(6.2, 2.1)):
    """Cumplimiento del mes en curso por canal de venta; el menor resaltado y la linea del cumplimiento general."""
    t = cv_t.iloc[::-1]
    fig, ax = plt.subplots(figsize=figsize)
    colores = [color if c_ == peor else "#c9c8c3" for c_ in t.index]
    ax.barh(t.index, t["cumpl"], color=colores, height=0.6)
    for i, (c_, fila) in enumerate(t.iterrows()):
        ax.annotate(f"{pct(fila['cumpl'])}  ({variacion(fila['delta'], True)})", (fila["cumpl"], i), xytext=(4, 0),
                    textcoords="offset points", va="center", fontsize=8.5,
                    color="#0b0b0b" if c_ == peor else "#52514e", fontweight="bold" if c_ == peor else "normal")
    ax.axvline(general, color="#52514e", linestyle="--", linewidth=1)
    ax.set_xlim(min(t["cumpl"].min(), general) - 8, max(t["cumpl"].max(), general) + 9)
    ejes_limpios(ax, formato_x=lambda v, _: f"{v:,.0f}%")
    ax.tick_params(axis="y", labelsize=9)
    ax.set_title(f"% cumplimiento por canal de venta · {ETQ_ACT} (línea: general {pct(general)})", loc="left",
                 fontsize=9, color="#52514e")
    plt.tight_layout()
    return fig


def caja(d, x, y, w, h, fondo, borde=None):
    forma = d.shapes.add_shape(1, x, y, w, h)
    forma.fill.solid()
    forma.fill.fore_color.rgb = fondo
    if borde is None:
        forma.line.fill.background()
    else:
        forma.line.color.rgb = borde
    forma.shadow.inherit = False
    return forma


def panel_region(d, x, y, w, vista):
    """La mitad de la lamina de una region: cumplimiento general, canales de venta, ruta del menor cumplimiento y el
    cumplimiento por tipo de descuento."""
    soc = VISTAS[vista]
    gen = resumen({"Sociedad": soc}).iloc[0]
    cv_t, (cv, cv_f), (cd, cd_f), (g, g_f) = ruta(soc)
    color = COLOR_SOC[soc]
    caja(d, x, y, w, Inches(0.06), rgb(color))
    caja_texto(d, x, y + Inches(0.1), w, Inches(0.35), vista.upper(), tam=14, color=rgb(color), negrita=True)
    # 2. cumplimiento general a corte
    caja_texto(d, x, y + Inches(0.45), Inches(2.1), Inches(0.8), pct(gen["cumpl"]), tam=40, negrita=True)
    kpi = caja_texto(d, x + Inches(2.15), y + Inches(0.52), w - Inches(2.15), Inches(0.8), [
        [("% cumplimiento del precio de lista a corte ", False), (f"{FECHA_MAX_D:%d/%m}", True)],
        [(f"{variacion(gen['delta'], True)} ", True), (f"vs {ETQ_ANT} ({pct(gen['cumpl_ant'])})", False)],
        [(f"Ajuste general {ETQ_ACT}: {millones(gen['ajuste'])} de {millones(gen['bruta'])} a lista", False)],
    ], tam=10.5, color=TINTA_2)
    kpi.text_frame.paragraphs[1].runs[0].font.color.rgb = MEJORA if gen["delta"] >= 0 else BAJA
    # 1. donde se cumple menos: canal de venta
    imagen(d, figura_canales(cv_t, cv, gen["cumpl"], color), x=x, y=y + Inches(1.35), w=w, h=Inches(2.05))
    # 1. ruta: canal de venta -> canal de distribucion -> grupo de material 6
    y_r = y + Inches(3.5)
    caja_texto(d, x, y_r, w, Inches(0.3), "DÓNDE SE CUMPLE MENOS", tam=10, color=TINTA_3, negrita=True)
    ancho, sep = (w - Inches(0.3)) / 3, Inches(0.15)
    for i, (nivel, nombre, f) in enumerate((("Canal de venta", cv, cv_f), ("Canal de distribución", cd, cd_f),
                                            ("Grupo de material 6", g, g_f))):
        x_i = x + i * (ancho + sep)
        caja(d, x_i, y_r + Inches(0.3), ancho, Inches(1.2), FONDO_RUTA)
        caja(d, x_i, y_r + Inches(0.3), Inches(0.05), Inches(1.2), rgb(color))
        paso = caja_texto(d, x_i + Inches(0.1), y_r + Inches(0.33), ancho - Inches(0.12), Inches(1.15), [
            [(f"{i + 1}. {nivel}", False)],
            [(str(nombre).capitalize()[:34], True)],
            [(pct(f["cumpl"]), True), (f"  {variacion(f['delta'], True)} vs {ETQ_ANT[:3]}", False)],
            [(f"ajuste {millones(f['ajuste'])}" + ("" if i == 0 else f" · {num(f['part'], 0)}% de la venta"), False)],
        ], tam=10, color=TINTA)
        partes = paso.text_frame.paragraphs
        partes[0].runs[0].font.size, partes[0].runs[0].font.color.rgb = Pt(8.5), TINTA_3
        partes[2].runs[1].font.color.rgb = MEJORA if f["delta"] >= 0 else BAJA
        partes[3].runs[0].font.size, partes[3].runs[0].font.color.rgb = Pt(8.5), TINTA_2
    # cumplimiento del precio de lista por tipo de descuento
    y_a = y_r + Inches(1.62)
    ancho_t = (w - Inches(0.15)) / 2
    for i, (nombre, act, ant, tono) in enumerate(cumplimiento_por_tipo(soc)):
        x_t = x + i * (ancho_t + Inches(0.15))
        caja(d, x_t, y_a, ancho_t, Inches(0.72), FONDO_RUTA)
        caja(d, x_t, y_a, Inches(0.06), Inches(0.72), rgb(tono))
        tipo = caja_texto(d, x_t + Inches(0.12), y_a + Inches(0.03), ancho_t - Inches(0.15), Inches(0.68), [
            [(f"% cumplimiento · {nombre}", False)],
            [(pct(act), True), (f"  {variacion(act - ant, True)}", True), (f" vs {ETQ_ANT}", False)],
        ], tam=10)
        p = tipo.text_frame.paragraphs
        p[0].runs[0].font.size, p[0].runs[0].font.color.rgb = Pt(9), TINTA_3
        p[1].runs[0].font.size = Pt(16)
        p[1].runs[1].font.color.rgb = MEJORA if act >= ant else BAJA
        p[1].runs[2].font.color.rgb = TINTA_2


def reporte_ejecutivo():
    nueva_presentacion()
    d = prs.slides.add_slide(prs.slide_layouts[6])
    total = resumen({}).iloc[0]
    caja_texto(d, MARGEN, Inches(0.22), UTIL_W, Inches(0.5),
               f"Cumplimiento del precio de lista · corte {FECHA_MAX_D:%d/%m/%Y}", tam=24, negrita=True)
    sub = caja_texto(d, MARGEN, Inches(0.72), UTIL_W, Inches(0.4), [
        [("Total compañía ", False), (pct(total["cumpl"]), True), (f" ({variacion(total['delta'], True)} vs {ETQ_ANT}) ", True),
         (f"· {ETQ_ACT} vs {ETQ_ANT} completo · ajuste general {millones(total['ajuste'])}", False)]], tam=13, color=TINTA_2)
    sub.text_frame.paragraphs[0].runs[2].font.color.rgb = MEJORA if total["delta"] >= 0 else BAJA
    ancho = (UTIL_W - Inches(0.4)) / 2
    for i, vista in enumerate(VISTAS):
        panel_region(d, MARGEN + i * (ancho + Inches(0.4)), Inches(1.25), ancho, vista)
    caja_texto(d, MARGEN, Inches(7.08), UTIL_W, Inches(0.3),
               f"{NOTA_CUMPLIMIENTO}. Canal de distribución y grupo "
               f"de material 6: el de menor cumplimiento entre los que pesan al menos {PARTICIPACION_MIN}% de la venta "
               f"de su nivel. Datos a {FECHA_MAX_D:%d/%m/%Y}.", tam=8, color=TINTA_3)
    guardar(nombre_informe())


reporte_ejecutivo()

Informe Diario Cumplimiento precio lista Sep 2026 vs Oct 2026 al 26.pptx: 1 láminas


## C.3 Una lámina por región

Más detalle de cada región, en una lámina: **cómo voy** (cumplimiento general y su variación) y el **tipo de
ajuste de precio** (ajuste de precio de mercado y fecha corta, en % de la lista, más las devoluciones); **dónde**
(mapa de calor canal de venta × canal de distribución, con la misma escala de color en las dos regiones: centrada
en el cumplimiento general de la compañía, rojo por debajo y verde por encima; en cada celda, en dos líneas, el
% de cumplimiento con su variación en puntos y el **precio neto por kg** acumulado del mes (valor neto / kilos
netos) con su diferencia en pesos, ambos contra el mes anterior completo, y fuera del mapa el **total** de cada canal de venta, de cada canal de distribución y de la región) y la
**tendencia**: el cumplimiento acumulado del mes, día a día, del mes en curso contra el mes anterior, con la
comparación a igual día del mes.

La **lámina 2** es el día a día: el **% de cumplimiento de cada día** (con el acumulado de cada mes en línea
punteada y la variación del acumulado a igual día del mes) y la **venta neta** y la **venta en kg** de cada
día (total de la región, descontadas las devoluciones: `Venta Neta` y `Venta Neta Kg`) en línea de tiempo, del 1 del mes anterior al corte (el eje llega al fin
del mes en curso), con los valores completos (sin redondear a millones) y el acumulado del mes día a día en línea
punteada. Las tres gráficas llevan la variación del mes en curso contra el mes anterior a igual día del mes, en la
nota del título y en la etiqueta del acumulado al final de la gráfica. El eje lleva el día y el nombre corto del
día de la semana (`1-Jue`).

In [27]:
ESCALA_MAPA = 6   # puntos de cumplimiento, a cada lado del general de la compania, en que el color del mapa satura
DIAS_SEMANA = ["Lun", "Mar", "Mié", "Jue", "Vie", "Sáb", "Dom"]   # nombre corto del dia (lunes = 0, como weekday())


def resumen_niveles(filtro: dict, niveles: list) -> pd.DataFrame:
    """Como resumen(), abierto por varios niveles a la vez y con los dos motivos de ajuste (en % de la lista)."""
    b = base
    for nivel, valor in filtro.items():
        b = b[b.index.get_level_values(nivel) == valor]
    ind = indicadores(b.groupby(level=[*niveles, "Periodo"]).sum())
    act = ind[ind.index.get_level_values("Periodo") == ETQ_ACT].droplevel("Periodo")
    ant = ind[ind.index.get_level_values("Periodo") == ETQ_ANT].droplevel("Periodo").reindex(act.index)
    t = pd.DataFrame({"cumpl": act[L_GEN], "cumpl_ant": ant[L_GEN], "bruta": act[VENTA_BRUTA], "ajuste": act[P_GEN],
                      "d_aj": act[D_AJ], "d_aj_ant": ant[D_AJ], "p_aj": act[P_AJ],
                      "d_fc": act[D_FC], "d_fc_ant": ant[D_FC], "p_fc": act[P_FC],
                      "precio": act[PRECIO_NETO], "precio_ant": ant[PRECIO_NETO]})
    t["delta"] = t["cumpl"] - t["cumpl_ant"]
    t["d_precio"] = t["precio"] - t["precio_ant"]
    return t


def variacion_valor(v, antes=r"\$", despues="") -> str:
    """+$200 o -12.345 kg (diferencia en valor completo). antes: \\$ para que matplotlib no lo lea como mathtext."""
    return "-" if pd.isna(v) else ("+" if v >= 0 else "-") + antes + num(abs(v), 0) + despues


def cambio_valor(act, ant, antes=r"\$", despues="") -> str:
    """-$550.215.402 (-0,4%): diferencia del mes en curso contra el mes anterior, en valor y en %."""
    return f"{variacion_valor(act - ant, antes, despues)} ({variacion((act / ant - 1) * 100 if ant else float('nan'), False)})"


def figura_mapa(soc, centro, figsize=(9.4, 4.0)):
    """Mapa de calor del % de cumplimiento del mes en curso: canal de venta x canal de distribucion (los que pesan al
    menos 1% de la venta de la region), con la variacion en puntos y el precio neto por kg con su diferencia en pesos,
    ambos contra el mes anterior completo. Fuera del mapa, el total de cada canal de venta (derecha), de cada canal de distribucion
    (abajo) y de la region (esquina), con todos sus canales (tambien los que no salen en el mapa). centro: el
    cumplimiento general de la compania, igual para las dos regiones, para que un mismo % tenga el mismo color en
    ambas. Rojo, por debajo del centro; verde, por encima; satura a ESCALA_MAPA puntos."""
    t = resumen_niveles({"Sociedad": soc}, ["Canal de Venta", "Canal de Distribucion"])
    peso = t["bruta"].groupby(level="Canal de Distribucion").sum()
    cds = list(peso[peso / peso.sum() * 100 >= 1].sort_values(ascending=False).index)
    cumpl = t["cumpl"].unstack().reindex(index=CANALES_VENTA_INFORME, columns=cds)
    delta = t["delta"].unstack().reindex(index=CANALES_VENTA_INFORME, columns=cds)
    precio = t["precio"].unstack().reindex(index=CANALES_VENTA_INFORME, columns=cds)
    d_precio = t["d_precio"].unstack().reindex(index=CANALES_VENTA_INFORME, columns=cds)
    # totales (fuera del mapa)
    filtro = {"Sociedad": soc}
    tot_cv = resumen_niveles(filtro, ["Canal de Venta"]).reindex(CANALES_VENTA_INFORME)
    tot_cd = resumen_niveles(filtro, ["Canal de Distribucion"]).reindex(cds)
    tot = resumen_niveles(filtro, ["Sociedad"]).iloc[0]
    n_f, n_c = cumpl.shape

    def texto_celda(v, d_pts, p, d_p):
        """Dos lineas: % cumplimiento y su variacion en puntos; precio neto por kg y su diferencia en pesos. Ambos
        contra el mes anterior completo."""
        if pd.isna(v):
            return "sin venta"
        if pd.isna(d_pts):
            return f"{pct(v)} sin venta en {ETQ_ANT[:3]}\n\\${num(p, 0)}"
        return f"{pct(v)} {variacion(d_pts, True)} vs {ETQ_ANT[:3]}\n\\${num(p, 0)} {variacion_valor(d_p)} vs {ETQ_ANT[:3]}"

    fig, ax = plt.subplots(figsize=figsize)
    ax.imshow(cumpl.values - centro, cmap="RdYlGn", vmin=-ESCALA_MAPA, vmax=ESCALA_MAPA, aspect="auto")
    for i in range(n_f):
        for j in range(n_c):
            v = cumpl.iat[i, j]
            texto = texto_celda(v, delta.iat[i, j], precio.iat[i, j], d_precio.iat[i, j])
            oscura = pd.notna(v) and abs(v - centro) >= 0.75 * ESCALA_MAPA   # extremos: fondo oscuro, texto blanco
            ax.text(j, i, texto, ha="center", va="center", fontsize=9.5, color="white" if oscura else "#0b0b0b",
                    fontweight="bold" if pd.notna(v) and v < centro else "normal")
    # totales: una columna a la derecha (canal de venta), una fila abajo (canal de distribucion) y la esquina (region)
    x_t, y_t = n_c + 0.1, n_f + 0.1
    celdas_total = [(x_t, i, tot_cv.iloc[i]) for i in range(n_f)]
    celdas_total += [(j, y_t, tot_cd.iloc[j]) for j in range(n_c)]
    celdas_total.append((x_t, y_t, tot))
    for x_, y_, f in celdas_total:
        ax.add_patch(plt.Rectangle((x_ - 0.47, y_ - 0.45), 0.94, 0.9, facecolor="#f6f5f2", edgecolor="none"))
        ax.text(x_, y_, texto_celda(f["cumpl"], f["delta"], f["precio"], f["d_precio"]), ha="center", va="center",
                fontsize=9.5,
                color="#0b0b0b", fontweight="bold")
    ax.set_xticks([*range(n_c), x_t], [*cds, "Total"], fontsize=10)
    ax.set_yticks([*range(n_f), y_t], [*cumpl.index, "Total"], fontsize=10)
    ax.set_xlim(-0.5, x_t + 0.5)
    ax.set_ylim(y_t + 0.5, -0.5)
    ax.tick_params(length=0)
    ax.xaxis.tick_top()
    ax.set_xlabel(f"En cada celda: % cumplimiento {ETQ_ACT} y su variación en puntos; precio neto por kg (valor neto / "
                  f"kilos netos) y su diferencia en pesos. Ambos vs {ETQ_ANT} completo",
                  fontsize=8.5, color="#52514e")
    for lado in ax.spines.values():
        lado.set_visible(False)
    plt.tight_layout()
    return fig


def figura_tendencia(soc, color, figsize=(12.4, 2.25), ax=None):
    """% cumplimiento del precio de lista ACUMULADO del mes, dia a dia (sumas desde el dia 1 hasta cada dia): el mes en
    curso contra el mes anterior, por dia del mes. Devuelve la figura y los dos acumulados a igual dia del mes.
    ax: eje donde se dibuja (None = figura nueva)."""
    d = diario[diario.index.get_level_values("Sociedad") == soc].droplevel("Sociedad").sort_index()
    if ax is None:
        fig, ax = plt.subplots(figsize=figsize)
    else:
        fig = ax.figure
    acumulados = {}
    for inicio, etq, color_, grosor in ((INICIO_ANT, ETQ_ANT, "#b5b4af", 2), (INICIO_ACT, ETQ_ACT, color, 2.8)):
        m = d[(d.index >= pd.Timestamp(inicio)) & (d.index < pd.Timestamp(inicio) + pd.offsets.MonthBegin(1))]
        acum = indicadores(m.cumsum())[L_GEN]
        ax.plot(acum.index.day, acum.values, color=color_, linewidth=grosor, marker="o", markersize=3.5,
                label=f"{etq} (acumulado)")
        ax.annotate(pct(acum.iloc[-1]), (acum.index[-1].day, acum.iloc[-1]), xytext=(6, 0), textcoords="offset points",
                    va="center", fontsize=9.5, color=color_, fontweight="bold")
        acumulados[etq] = acum
    dia = FECHA_MAX_D.day
    ant = acumulados[ETQ_ANT]
    ant_dia = ant[ant.index.day <= dia].iloc[-1]
    ax.axvline(dia, color="#8c8b87", linestyle=":", linewidth=1)
    ax.set_xticks(range(1, 32))
    ax.set_xlim(0.5, 32)
    ax.margins(y=0.2)
    ejes_limpios(ax, formato_y=lambda v, _: f"{v:,.0f}%")
    ax.set_xlabel("día del mes", fontsize=8, color="#52514e")
    ax.legend(frameon=False, fontsize=9, loc="lower right", ncols=2)
    plt.tight_layout()
    return fig, acumulados[ETQ_ACT].iloc[-1], ant_dia


def color_delta(v, mejor_si_sube=True):
    return MEJORA if (v >= 0) == mejor_si_sube else BAJA


def dias_region(soc) -> pd.DataFrame:
    """diario de la region, del 1 del mes anterior al fin del mes en curso."""
    fin_mes = pd.Timestamp(INICIO_ACT) + pd.offsets.MonthBegin(1)   # EXCLUSIVO
    d = diario[diario.index.get_level_values("Sociedad") == soc].droplevel("Sociedad").sort_index()
    return d[(d.index >= pd.Timestamp(INICIO_ANT)) & (d.index < fin_mes)]


def eje_linea_tiempo(ax):
    """Eje x en linea de tiempo, del 1 del mes anterior al fin del mes en curso: dia y nombre corto del dia (1-Jue),
    cambio de mes, dia del corte y nombre de cada mes. Devuelve la posicion de cada dia en el eje."""
    fin_mes = pd.Timestamp(INICIO_ACT) + pd.offsets.MonthBegin(1)   # EXCLUSIVO
    dias = pd.date_range(INICIO_ANT, fin_mes - pd.Timedelta(days=1))
    pos = pd.Series(range(len(dias)), index=dias)                     # posicion de cada dia en el eje
    ax.set_xticks(range(len(dias)), [f"{f.day}-{DIAS_SEMANA[f.weekday()]}" for f in dias], fontsize=6, rotation=90)
    ax.set_xlim(-0.7, len(dias) - 0.3)
    ax.axvline(pos[pd.Timestamp(INICIO_ACT)] - 0.5, color="#52514e", linewidth=0.8)   # cambio de mes
    ax.axvline(pos[pd.Timestamp(FECHA_MAX_D).normalize()], color="#8c8b87", linestyle=":", linewidth=1)   # corte
    for inicio, texto in ((INICIO_ANT, f"{MES_ANT_TXT} {INICIO_ANT.year}"), (INICIO_ACT, f"{MES_ACT_TXT} {INICIO_ACT.year}")):
        ax.annotate(texto.capitalize(), (pos[pd.Timestamp(inicio)] - 0.4, 0), xycoords=ax.get_xaxis_transform(),
                    xytext=(0, -30), textcoords="offset points", va="top", fontsize=8, color="#52514e",
                    fontweight="bold")   # debajo de los rotulos verticales de los dias
    return pos


def etiqueta_acumulado(ax, x, y, texto: str, cambio, texto_cambio: str):
    """Al final de la grafica: el acumulado del mes en curso y, debajo, su variacion contra el mes anterior a igual dia
    del mes (verde si sube, rojo si baja)."""
    ax.annotate(texto, (x, y), xytext=(8, -2), textcoords="offset points", va="top", fontsize=8.5, color="#0b0b0b",
                fontweight="bold")
    ax.annotate(texto_cambio, (x, y), xytext=(8, -14), textcoords="offset points", va="top", fontsize=8.5,
                color="#1baf7a" if cambio >= 0 else "#d64545", fontweight="bold")


def barras_dia(ax, soc, medida, color, antes="", despues=""):
    """Barras de `medida` (suma de la region) dia a dia en linea de tiempo, del 1 del mes anterior
    al fin del mes en curso (sin barras despues del corte), con el valor completo de cada dia: gris el mes anterior,
    color el mes en curso. Encima, en un eje secundario, el acumulado de cada mes dia a dia (linea punteada) y la
    etiqueta del acumulado del mes en curso con su variacion contra el mes anterior a igual dia del mes
    (etiqueta_acumulado()). antes / despues: unidad del valor (\\$, " kg"). Devuelve la suma del mes anterior, la del
    mes en curso hasta el corte y la del mes anterior a igual dia del mes."""
    d = dias_region(soc)[medida]
    ejes_limpios(ax, formato_y=lambda v, _: num(v, 0))   # antes de eje_linea_tiempo(): fija el tamano de los rotulos
    pos = eje_linea_tiempo(ax)
    es_act = d.index >= pd.Timestamp(INICIO_ACT)
    for mascara, color_, tinta in ((~es_act, "#b5b4af", "#8c8b87"), (es_act, color, "#0b0b0b")):
        m = d[mascara]
        barras = ax.bar(pos[m.index], m.values, width=0.75, color=color_)
        ax.bar_label(barras, labels=[num(v, 0) for v in m.values], rotation=90, padding=2, fontsize=5, color=tinta)
    ax.set_ylim(min(0, d.min()), ax.get_ylim()[1] * 1.85)   # espacio para los rotulos verticales
    acum = d.groupby(es_act).cumsum()                         # suma desde el dia 1 de cada mes
    ax2 = ax.twinx()
    for mascara, color_ in ((~es_act, "#b5b4af"), (es_act, color)):
        m = acum[mascara]
        ax2.plot(pos[m.index], m.values, color=color_, linestyle="--", linewidth=1.2)
    ax2.set_ylim(min(0, acum.min()), acum.max() * 1.1)
    ax2.axis("off")
    ant = d[~es_act]
    ant_mes, act, ant_dia = ant.sum(), d[es_act].sum(), ant[ant.index.day <= FECHA_MAX_D.day].sum()
    etiqueta_acumulado(ax2, pos[d.index[es_act][-1]], act, f"acum. {antes}{num(act, 0)}{despues}", act - ant_dia,
                       cambio_valor(act, ant_dia, antes, despues))
    return ant_mes, act, ant_dia


def titulo_y_variacion(ax, titulo: str, cambio, texto_cambio=None):
    """Titulo del grafico y, debajo, la frase de variacion del mes en curso contra el mes anterior al mismo dia del
    mes: verde si sube, rojo si baja. texto_cambio: la variacion ya escrita (None = cambio en puntos)."""
    ax.set_title(titulo, loc="left", fontsize=9.5, color="#52514e", pad=16)
    ax.text(0, 1.02, f"Variación {ETQ_ACT} vs {ETQ_ANT[:3]} al {FECHA_MAX_D.day}: "
                     f"{texto_cambio or variacion(cambio, True)}",
            transform=ax.transAxes, va="bottom", fontsize=9.5, fontweight="bold",
            color="#1baf7a" if cambio >= 0 else "#d64545")


def linea_cumplimiento_dia(ax, soc, color):
    """% cumplimiento del precio de lista de cada dia (desde las sumas del dia) en linea de tiempo, gris el mes anterior
    y color el mes en curso, con el acumulado de cada mes en linea punteada. Titulo: los dos acumulados y la variacion
    del acumulado del mes en curso contra el del mes anterior a igual dia del mes (titulo_y_variacion() y, al final de
    la grafica, etiqueta_acumulado())."""
    d = dias_region(soc)
    ejes_limpios(ax, formato_y=lambda v, _: f"{v:,.0f}%")   # antes de eje_linea_tiempo(): fija el tamano de los rotulos
    pos = eje_linea_tiempo(ax)
    dia = FECHA_MAX_D.day
    es_act = d.index >= pd.Timestamp(INICIO_ACT)
    por_dia = indicadores(d)[L_GEN]
    acum = {}
    for mascara, etq, color_, tinta in ((~es_act, ETQ_ANT, "#b5b4af", "#8c8b87"), (es_act, ETQ_ACT, color, "#0b0b0b")):
        m, p = d[mascara], por_dia[mascara].dropna()
        x = pos[p.index]
        ax.plot(x, p.values, color=color_, linewidth=2.5, marker="o", markersize=4)
        for x_, v in zip(x, p.values):
            ax.annotate(pct(v), (x_, v), xytext=(0, 5), textcoords="offset points", rotation=90, ha="center",
                        va="bottom", fontsize=5.5, color=tinta)
        acum[etq] = indicadores(m.sum().to_frame().T)[L_GEN].iat[0]   # acumulado del mes, desde sumas
        ax.hlines(acum[etq], x.min() - 0.3, x.max() + 0.3, colors=color_, linestyles="--", linewidth=1.2)
    ant = d[~es_act]
    ant_dia = indicadores(ant[ant.index.day <= dia].sum().to_frame().T)[L_GEN].iat[0]   # mes anterior a igual dia
    cambio = acum[ETQ_ACT] - ant_dia
    etiqueta_acumulado(ax, pos[pd.Timestamp(FECHA_MAX_D).normalize()], acum[ETQ_ACT], f"acum. {pct(acum[ETQ_ACT])}",
                       cambio, variacion(cambio, True))
    lo, hi = por_dia.min(), por_dia.max()
    ax.set_ylim(lo - (hi - lo) * 0.1, hi + (hi - lo) * 0.3)   # espacio para los rotulos verticales
    titulo_y_variacion(ax, f"% cumplimiento del precio de lista por día · {INICIO_ANT:%d/%m} al {FECHA_MAX_D:%d/%m} · "
                           f"acumulado {ETQ_ANT}: {pct(acum[ETQ_ANT])} · acumulado {ETQ_ACT}: {pct(acum[ETQ_ACT])}",
                       cambio)


def figura_dia_a_dia(soc, color, figsize=(12.4, 7.0)):
    """Lamina 2: el % cumplimiento de cada dia (linea_cumplimiento_dia()) y la venta neta y en kg dia a dia (barras_dia()),
    del 1 del mes anterior al corte (eje hasta el fin del mes en curso), cada una con su acumulado y su variacion contra
    el mes anterior a igual dia del mes."""
    fig, (ax_t, ax_n, ax_k) = plt.subplots(3, 1, figsize=figsize)
    linea_cumplimiento_dia(ax_t, soc, color)
    for ax, medida, nombre, antes, despues in ((ax_n, "Venta Neta", "Valor venta neta", r"\$", ""),   # \$: sin mathtext
                                               (ax_k, "Venta Neta Kg", "Venta neta kg", "", " kg")):
        ant_mes, act, ant_dia = barras_dia(ax, soc, medida, color, antes, despues)
        titulo_y_variacion(ax, f"{nombre} día a día · {INICIO_ANT:%d/%m} al {FECHA_MAX_D:%d/%m} · acumulado {ETQ_ANT}: "
                               f"{antes}{num(ant_mes, 0)}{despues} · acumulado {ETQ_ACT}: {antes}{num(act, 0)}{despues}",
                           act - ant_dia, cambio_valor(act, ant_dia, antes, despues))
    plt.tight_layout()
    return fig


def lamina_dia_a_dia(vista: str):
    """Lamina 2 de la region, en la presentacion en construccion: el dia a dia del mes (figura_dia_a_dia())."""
    soc = VISTAS[vista]
    color = COLOR_SOC[soc]
    d = prs.slides.add_slide(prs.slide_layouts[6])
    caja(d, 0, 0, ANCHO, Inches(0.08), rgb(color))
    caja_texto(d, MARGEN, Inches(0.2), UTIL_W, Inches(0.5),
               f"{vista} · día a día del mes · corte {FECHA_MAX_D:%d/%m/%Y}", tam=22, negrita=True)
    caja_texto(d, MARGEN, Inches(0.68), UTIL_W, Inches(0.35),
               f"{ETQ_ACT} vs {ETQ_ANT} completo · valor y kilos netos (descontadas las devoluciones) · valores "
               f"completos", tam=12, color=TINTA_2)
    imagen(d, figura_dia_a_dia(soc, color), x=MARGEN, y=Inches(1.1), w=UTIL_W, h=Inches(5.95))
    caja_texto(d, MARGEN, Inches(7.08), UTIL_W, Inches(0.3),
               f"{NOTA_CUMPLIMIENTO}; por día y, en línea punteada "
               f"horizontal, el acumulado del mes. Valor venta neta ($) y venta neta kg: total de la región por día, venta menos devoluciones, del "
               f"{INICIO_ANT:%d/%m} al corte, y en línea punteada el acumulado del mes día a día; gris: {ETQ_ANT}, color: {ETQ_ACT}. "
               f"Variación: mes en curso vs {ETQ_ANT[:3]} a igual día del mes. Línea punteada vertical: día del corte. Datos a {FECHA_MAX_D:%d/%m/%Y}.", tam=8, color=TINTA_3)


def reporte_region(vista: str):
    """Una lamina por region: como voy y tipo de ajuste de precio, donde (canal de venta x canal de distribucion) y la
    tendencia del cumplimiento acumulado, mes en curso vs mes anterior."""
    soc = VISTAS[vista]
    color = COLOR_SOC[soc]
    gen = resumen_niveles({"Sociedad": soc}, ["Sociedad"]).iloc[0]
    general_cia = resumen({}).iloc[0]["cumpl"]   # centro de la escala del mapa, el mismo en las dos regiones
    sumas = base[base.index.get_level_values("Sociedad") == soc].groupby(level="Periodo").sum()
    devuelto, neta = sumas.loc[ETQ_ACT, "Devolucion Valor"], sumas.loc[ETQ_ACT, "Venta Neta"]
    nueva_presentacion()
    d = prs.slides.add_slide(prs.slide_layouts[6])
    caja(d, 0, 0, ANCHO, Inches(0.08), rgb(color))
    caja_texto(d, MARGEN, Inches(0.2), UTIL_W, Inches(0.5),
               f"{vista} · cumplimiento del precio de lista · corte {FECHA_MAX_D:%d/%m/%Y}", tam=22, negrita=True)
    caja_texto(d, MARGEN, Inches(0.68), UTIL_W, Inches(0.35),
               f"{ETQ_ACT} vs {ETQ_ANT} completo · cumplimiento = precio neto / precio de lista", tam=12,
               color=TINTA_2)

    # ------------------------------------------------------------------ izquierda: como voy y tipo de ajuste de precio
    x, w = MARGEN, Inches(3.9)
    caja_texto(d, x, Inches(1.15), w, Inches(0.3), "CÓMO VOY", tam=10, color=TINTA_3, negrita=True)
    caja_texto(d, x, Inches(1.38), w, Inches(0.85), pct(gen["cumpl"]), tam=44, negrita=True, color=rgb(color))
    kpi = caja_texto(d, x, Inches(2.25), w, Inches(0.55), [
        [(f"{variacion(gen['delta'], True)} ", True), (f"vs {ETQ_ANT} ({pct(gen['cumpl_ant'])})", False)],
        [(f"Ajuste {millones(gen['ajuste'])} de {millones(gen['bruta'])} a lista", False)],
    ], tam=11, color=TINTA_2)
    kpi.text_frame.paragraphs[0].runs[0].font.color.rgb = color_delta(gen["delta"])
    caja_texto(d, x, Inches(2.95), w, Inches(0.3), "TIPO DE AJUSTE DE PRECIO", tam=10, color=TINTA_3,
               negrita=True)
    for i, (nombre, pts, ant, plata, tono) in enumerate((
            ("Ajuste de precio de mercado", gen["d_aj"], gen["d_aj_ant"], gen["p_aj"], "#1baf7a"),
            ("Fecha corta", gen["d_fc"], gen["d_fc_ant"], gen["p_fc"], "#eda100"))):
        caja(d, x, Inches(3.3 + i * 0.5), Inches(0.12), Inches(0.36), rgb(tono))
        linea = caja_texto(d, x + Inches(0.2), Inches(3.24 + i * 0.5), w - Inches(0.2), Inches(0.5), [
            [(f"{nombre} ", False), (pct(pts), True)],
            [(f"{variacion(pts - ant, True)} vs {ETQ_ANT[:3]}", True), (f" · {millones(plata)}", False)],
        ], tam=10.5, color=TINTA)
        segunda = linea.text_frame.paragraphs[1].runs
        segunda[0].font.color.rgb = color_delta(pts - ant, mejor_si_sube=False)   # menos ajuste es mejor
        segunda[0].font.size = segunda[1].font.size = Pt(9.5)
    caja_texto(d, x, Inches(4.3), w, Inches(0.3), [
        [("Devoluciones: ", True), (f"{millones(devuelto)} ({num(devuelto / neta * 100)}% de la venta neta)", False)]],
        tam=10, color=TINTA_2)

    # ------------------------------------------------------------------ derecha: donde
    xr = MARGEN + Inches(4.2)
    wr = UTIL_W - Inches(4.2)
    caja_texto(d, xr, Inches(1.15), wr, Inches(0.3),
               f"DÓNDE: % CUMPLIMIENTO POR CANAL DE VENTA Y DE DISTRIBUCIÓN · GENERAL {pct(gen['cumpl'])}", tam=10,
               color=TINTA_3, negrita=True)
    imagen(d, figura_mapa(soc, general_cia), x=xr, y=Inches(1.45), w=wr, h=Inches(3.1))

    # ------------------------------------------------------------------ abajo: tendencia del mes vs el anterior
    fig, act, ant_dia = figura_tendencia(soc, color)
    tendencia = caja_texto(d, MARGEN, Inches(4.72), UTIL_W, Inches(0.3), [
        [("TENDENCIA: % CUMPLIMIENTO ACUMULADO DEL MES, DÍA A DÍA · ", True),
         (f"al día {FECHA_MAX_D.day}: {ETQ_ACT[:3]} {pct(act)} vs {ETQ_ANT[:3]} {pct(ant_dia)} ", True),
         (f"({variacion(act - ant_dia, True)})", True)]], tam=10, color=TINTA_3)
    tendencia.text_frame.paragraphs[0].runs[1].font.color.rgb = TINTA
    tendencia.text_frame.paragraphs[0].runs[2].font.color.rgb = color_delta(act - ant_dia)
    imagen(d, fig, x=MARGEN, y=Inches(5.0), w=UTIL_W, h=Inches(2.05))
    caja_texto(d, MARGEN, Inches(7.08), UTIL_W, Inches(0.3),
               f"{NOTA_CUMPLIMIENTO}. Acumulado: con las sumas del "
               f"día 1 al día de cada punto. Mapa: misma escala de color en las dos regiones, centrada en el general de la "
               f"compañía ({pct(general_cia)}): rojo por debajo, verde por encima. Datos a {FECHA_MAX_D:%d/%m/%Y}.",
               tam=8, color=TINTA_3)
    lamina_dia_a_dia(vista)
    guardar(nombre_informe(vista))


for vista in VISTAS:
    reporte_region(vista)

Informe Diario Cumplimiento precio lista Región Oriente Sep 2026 vs Oct 2026 al 26.pptx: 2 láminas
Informe Diario Cumplimiento precio lista Región Occidente Sep 2026 vs Oct 2026 al 26.pptx: 2 láminas
